# 06 · Fine-tuning Whisper on our Sinhala farming phrases (Member 4: chatbot)

**Status: NOT RUN YET. It waits for the team's recordings.** Until then the chatbot uses the pretrained
Whisper-Sinhala model as it is (`STT_MODEL=small` in `.env`), and this notebook stops after the data check.

What it does once the recordings exist:
1. **Data.** The team reads the 249 prompts in `eval/voice/record_prompts.csv` with `eval/voice/recorder.html`
   (2–3 speakers or more), and `python -m member4_chatbot.pipeline.voice_recordings` imports them. Each prompt has a
   fixed split: `finetune` phrases train, `test` phrases are held out. Recordings of a test phrase never train.
2. **Mix with general Sinhala.** A fixed sample of OpenSLR-52 (Google's Sinhala ASR corpus, CC BY-SA 4.0) is mixed
   in, so the model does not forget general Sinhala while it learns farming words. Farming clips are repeated
   (`FARM_REPEAT`) so they are not drowned out.
3. **Fine-tune** the small Whisper-Sinhala model (`BASE`) with the Hugging Face `Seq2SeqTrainer`. On a free Colab T4
   (fp16), about 20–40 minutes for the default settings. On CPU it is too slow; use Colab or a CUDA GPU.
4. **Report** WER and CER on the held-out **test phrases**, before vs after, each with and without the farming-term
   dictionary (`data/voice/stt_corrections.yaml`), plus a check on OpenSLR-30 (general Sinhala) that the model has
   not forgotten general speech.
5. **Save** to `artifacts/stt/finetuned/` with `metrics.json`. Set `STT_MODEL=finetuned` in `.env` to use it.
   The last cell says honestly whether the fine-tuned model beats the base model on farming phrases.

**Run on Colab (T4):** run `python -m member4_chatbot.pipeline.voice_recordings --pack` on the laptop, upload
`eval/voice/recordings/colab_pack.zip` and this notebook to Colab, choose *Runtime → T4 GPU*, run all. Download
`finetuned_model.zip` at the end and unzip it into `backend/member4-chatbot/artifacts/stt/finetuned/`.
**Run locally:** from `backend/member4-chatbot/notebooks/` with a CUDA build of torch.

In [1]:
RUN_FINETUNE = True          # False: only show the data check and any saved metrics
SEED = 20261019
BASE = "seniruk/whisper-small-si"  # the small model chosen in notebook 07 (STT_MODEL=small)
MIN_FINETUNE_CLIPS, MIN_TEST_CLIPS = 150, 40
OPENSLR52_N = 3000           # general-Sinhala utterances mixed in (from shard 0 of OpenSLR-52)
FARM_REPEAT = 3              # each farming clip is seen this many times per epoch
EPOCHS, LR, BATCH, WARMUP_STEPS = 4, 1e-5, 16, 100

In [2]:
import sys, os, json, time, random, zipfile, io, urllib.request
from pathlib import Path
HERE = Path.cwd()
if (HERE.parent / "member4_chatbot").exists():          # inside the repository
    ROOT = HERE.parent
else:                                                    # Colab: unpack colab_pack.zip next to the notebook
    ROOT = HERE / "m4"
    pack = next(iter(list(HERE.glob("colab_pack.zip")) + list(Path("/content").glob("colab_pack.zip"))), None)
    if pack and not ROOT.exists():
        zipfile.ZipFile(pack).extractall(ROOT)
sys.path.insert(0, str(ROOT))
import numpy as np, pandas as pd, torch
from member4_chatbot.voice import scoring
from member4_chatbot.voice.corrections import correct, load_rules
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
REC = ROOT / "eval" / "voice" / "recordings"
OUT = ROOT / "artifacts" / "stt" / "finetuned"
print("root:", ROOT, "| device:", DEVICE, torch.cuda.get_device_name(0) if DEVICE == "cuda" else "")

root: E:\soilmind\backend\member4-chatbot | device: cpu 


## 1. The team's recordings

In [3]:
manifest = REC / "manifest.csv"
farm = pd.read_csv(manifest, keep_default_na=False) if manifest.exists() else pd.DataFrame(
    columns=["clip_id", "speaker", "prompt_id", "text", "split", "file", "duration_s"])
n_ft, n_test = (farm.split == "finetune").sum(), (farm.split == "test").sum()
print(f"recordings: {len(farm)} from {farm.speaker.nunique()} speakers; finetune {n_ft}, test {n_test}")
if len(farm):
    print(farm.groupby(["speaker", "split"]).size().unstack(fill_value=0))
READY = RUN_FINETUNE and n_ft >= MIN_FINETUNE_CLIPS and n_test >= MIN_TEST_CLIPS
if not READY:
    print(f"\nNOT RUN: needs at least {MIN_FINETUNE_CLIPS} finetune and {MIN_TEST_CLIPS} test recordings "
          f"(have {n_ft} / {n_test}). Record with eval/voice/recorder.html, then "
          "python -m member4_chatbot.pipeline.voice_recordings. The app keeps using the pretrained model.")
    saved = OUT / "metrics.json"
    if saved.exists():
        print("saved metrics from an earlier run:", json.loads(saved.read_text()))

recordings: 0 from 0 speakers; finetune 0, test 0

NOT RUN: needs at least 150 finetune and 40 test recordings (have 0 / 0). Record with eval/voice/recorder.html, then python -m member4_chatbot.pipeline.voice_recordings. The app keeps using the pretrained model.


## 2. Training mix: our finetune phrases + a fixed OpenSLR-52 sample

In [4]:
def openslr52_sample(n, cache=ROOT / "data" / "voice" / "external" / "openslr52"):
    """n utterances of OpenSLR-52 shard 0 (~915 MB download, kept in data/voice/external/)."""
    cache.mkdir(parents=True, exist_ok=True)
    base = "https://www.openslr.org/resources/52/"
    for name in ("utt_spk_text.tsv", "asr_sinhala_0.zip"):
        if not (cache / name).exists():
            print("downloading", name); urllib.request.urlretrieve(base + name, cache / name)
    meta = pd.read_csv(cache / "utt_spk_text.tsv", sep="\t", names=["utt", "spk", "text"], dtype=str)
    z = zipfile.ZipFile(cache / "asr_sinhala_0.zip")
    names = {Path(n).stem: n for n in z.namelist() if n.endswith(".flac")}
    meta = meta[meta.utt.isin(names)].sample(n=n, random_state=SEED)
    return [(z.read(names[u]), t) for u, t in zip(meta.utt, meta.text)]

if READY:
    from member4_chatbot.voice.audio import load_audio
    ft = farm[farm.split == "finetune"]
    farm_items = [(load_audio((ROOT / f).read_bytes())[0], t) for f, t in zip(ft.file, ft.text)] * FARM_REPEAT
    general = [(load_audio(b)[0], t) for b, t in openslr52_sample(OPENSLR52_N)]
    train_items = farm_items + general
    random.Random(SEED).shuffle(train_items)
    test = farm[farm.split == "test"].reset_index(drop=True)
    print(f"train: {len(farm_items)} farming (x{FARM_REPEAT}) + {len(general)} OpenSLR-52; test phrases: {len(test)}")

## 3. Before: the base model on the held-out test phrases

In [5]:
from transformers import WhisperForConditionalGeneration, WhisperProcessor

def transcribe(model, processor, files, batch=8):
    from member4_chatbot.voice.audio import load_audio
    out = []
    model.eval()
    for i in range(0, len(files), batch):
        audio = [load_audio((ROOT / f).read_bytes())[0] for f in files[i:i + batch]]
        x = processor(audio, sampling_rate=16000, return_tensors="pt", return_attention_mask=True)
        with torch.inference_mode():
            ids = model.generate(x.input_features.to(DEVICE, model.dtype), attention_mask=x.attention_mask.to(DEVICE),
                                 language="sinhala", task="transcribe", max_new_tokens=220)
        out += [t.strip() for t in processor.batch_decode(ids, skip_special_tokens=True)]
    return out

def scores(refs, hyps):
    rules = load_rules()
    fixed = [correct(h, rules, log_to_file=False)[0] for h in hyps]
    return {"wer": scoring.wer(refs, hyps), "cer": scoring.cer(refs, hyps),
            "wer_dict": scoring.wer(refs, fixed), "cer_dict": scoring.cer(refs, fixed), "n": len(refs)}

if READY:
    processor = WhisperProcessor.from_pretrained(BASE)
    base_model = WhisperForConditionalGeneration.from_pretrained(BASE).to(DEVICE)
    before_hyps = transcribe(base_model, processor, list(test.file))
    before = scores(list(test.text), before_hyps)
    print("base model on test phrases:", before)

## 4. Fine-tune

In [6]:
if READY:
    from dataclasses import dataclass
    from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments, set_seed
    set_seed(SEED)
    model = WhisperForConditionalGeneration.from_pretrained(BASE)
    model.generation_config.language, model.generation_config.task = "sinhala", "transcribe"
    model.generation_config.forced_decoder_ids = None
    processor.tokenizer.set_prefix_tokens(language="sinhala", task="transcribe")

    class Items(torch.utils.data.Dataset):
        def __init__(self, items): self.items = items
        def __len__(self): return len(self.items)
        def __getitem__(self, i):
            audio, text = self.items[i]
            feats = processor.feature_extractor(audio, sampling_rate=16000).input_features[0]
            return {"input_features": feats, "labels": processor.tokenizer(text).input_ids}

    @dataclass
    class Collator:
        def __call__(self, batch):
            feats = processor.feature_extractor.pad([{"input_features": b["input_features"]} for b in batch], return_tensors="pt")
            labels = processor.tokenizer.pad([{"input_ids": b["labels"]} for b in batch], return_tensors="pt")
            ids = labels["input_ids"].masked_fill(labels.attention_mask.ne(1), -100)
            if (ids[:, 0] == model.config.decoder_start_token_id).all():
                ids = ids[:, 1:]
            return {"input_features": feats["input_features"], "labels": ids}

    args = Seq2SeqTrainingArguments(
        output_dir=str(HERE / "_stt_runs"), per_device_train_batch_size=BATCH, learning_rate=LR,
        warmup_steps=WARMUP_STEPS, num_train_epochs=EPOCHS, fp16=DEVICE == "cuda", gradient_checkpointing=True,
        logging_steps=25, save_strategy="no", report_to=[], seed=SEED, dataloader_num_workers=0,
        remove_unused_columns=False)
    trainer = Seq2SeqTrainer(model=model, args=args, train_dataset=Items(train_items), data_collator=Collator())
    t0 = time.time(); trainer.train(); train_min = (time.time() - t0) / 60
    print(f"training took {train_min:.1f} min on {DEVICE}")

## 5. After: the fine-tuned model on the same test phrases, and a general-Sinhala check

In [7]:
if READY:
    model.to(DEVICE)
    after_hyps = transcribe(model, processor, list(test.file))
    after = scores(list(test.text), after_hyps)
    general_set = ROOT / "eval" / "voice" / "openslr30_test.csv"
    general = {}
    if general_set.exists():
        g = pd.read_csv(general_set, keep_default_na=False)
        general = {"base": scores(list(g.text), transcribe(base_model, processor, list(g.file))),
                   "finetuned": scores(list(g.text), transcribe(model, processor, list(g.file)))}
    table = pd.DataFrame({"base": before, "finetuned (ours)": after}).T[["n", "wer", "cer", "wer_dict", "cer_dict"]]
    print("held-out farming test phrases:"); print(table.round(4))
    if general:
        print("\nOpenSLR-30 (general Sinhala, forgetting check):"); print(pd.DataFrame(general).T.round(4))
    per_clip = test.assign(base=before_hyps, finetuned=after_hyps)
    per_clip.to_csv(HERE / "_stt_test_transcripts.csv", index=False)

## 6. Save, and the honest verdict

In [8]:
if READY:
    OUT.mkdir(parents=True, exist_ok=True)
    model.save_pretrained(OUT); processor.save_pretrained(OUT)
    better = after["cer"] < before["cer"]
    metrics = {"base_model": BASE, "device": DEVICE, "train_minutes": round(train_min, 1), "seed": SEED,
               "settings": {"epochs": EPOCHS, "lr": LR, "batch": BATCH, "farm_repeat": FARM_REPEAT,
                            "openslr52_n": OPENSLR52_N},
               "data": {"finetune_clips": int(n_ft), "test_clips": int(n_test), "speakers": int(farm.speaker.nunique())},
               "test_phrases": {"base": before, "finetuned": after}, "openslr30": general,
               "verdict": ("fine-tuned beats base on farming test phrases (CER)" if better else
                           "fine-tuned does NOT beat base on farming test phrases (CER); keep STT_MODEL=small")}
    (OUT / "metrics.json").write_text(json.dumps(metrics, indent=2, ensure_ascii=False))
    print(metrics["verdict"])
    if not (HERE.parent / "member4_chatbot").exists():   # Colab: zip the model for download
        import shutil; shutil.make_archive(str(HERE / "finetuned_model"), "zip", OUT); print("download finetuned_model.zip")